# 01 — Setup & Image Generation

This notebook:
1. Installs dependencies
2. Loads Flux-schnell with NF4 quantization
3. Generates mirror and non-mirror image pairs
4. Saves images and ROI definitions

In [ ]:
# Clone repo and set up paths (run once on Colab)
!git clone https://github.com/iker0/mi-mirror.git /content/mi-mirror 2>/dev/null || echo "Already cloned"
%cd /content/mi-mirror

In [ ]:
import sys
sys.path.insert(0, '/content/mi-mirror')

import torch
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

from scripts.config import (
    MIRROR_PROMPTS, NON_MIRROR_PROMPTS, SEEDS, PROMPT_OBJECTS,
    MIRROR_DIR, NON_MIRROR_DIR, ROI_DIR,
    MODEL_ID, RESOLUTION, NUM_INFERENCE_STEPS,
)
from scripts.model_loader import load_flux_pipeline
from scripts.roi import get_object_and_reflection_rois
from scripts.generate import generate_baseline, save_image

print(f"Model: {MODEL_ID}")
print(f"Resolution: {RESOLUTION}x{RESOLUTION}")
print(f"Steps: {NUM_INFERENCE_STEPS}")
print(f"Mirror prompts: {len(MIRROR_PROMPTS)}")
print(f"Non-mirror prompts: {len(NON_MIRROR_PROMPTS)}")
print(f"Seeds: {SEEDS}")
print(f"Total images: {(len(MIRROR_PROMPTS) + len(NON_MIRROR_PROMPTS)) * len(SEEDS)}")

In [ ]:
# Install dependencies (run once on Colab)
!pip install -q "diffusers>=0.31.0" transformers bitsandbytes accelerate sentencepiece matplotlib scipy

In [ ]:
# Authenticate with Hugging Face (required for gated models like Flux)
# Get your token at https://huggingface.co/settings/tokens
# Also accept model terms at https://huggingface.co/black-forest-labs/FLUX.1-schnell
from huggingface_hub import login
login()

In [ ]:
# Load model
pipe = load_flux_pipeline(quantize_nf4=False, cpu_offload=True)
print("Model loaded successfully!")

In [ ]:
# Generate mirror images
print("Generating mirror images...")
MIRROR_DIR.mkdir(parents=True, exist_ok=True)

for i, prompt in enumerate(MIRROR_PROMPTS):
    for seed in SEEDS:
        filename = f"prompt{i:02d}_seed{seed}.png"
        path = MIRROR_DIR / filename
        if path.exists():
            print(f"  Skipping (exists): {filename}")
            continue
        image = generate_baseline(pipe, prompt, seed)
        save_image(image, path)
        print(f"  Generated: {filename} — {prompt[:50]}...")

print(f"\nMirror images saved to {MIRROR_DIR}")

In [ ]:
# Generate non-mirror images
print("Generating non-mirror images...")
NON_MIRROR_DIR.mkdir(parents=True, exist_ok=True)

for i, prompt in enumerate(NON_MIRROR_PROMPTS):
    for seed in SEEDS:
        filename = f"prompt{i:02d}_seed{seed}.png"
        path = NON_MIRROR_DIR / filename
        if path.exists():
            print(f"  Skipping (exists): {filename}")
            continue
        image = generate_baseline(pipe, prompt, seed)
        save_image(image, path)
        print(f"  Generated: {filename} — {prompt[:50]}...")

print(f"\nNon-mirror images saved to {NON_MIRROR_DIR}")

In [ ]:
# Define and save per-image CLIPSeg ROIs
import json

ROI_DIR.mkdir(parents=True, exist_ok=True)
all_rois = {}

for i, obj_name in enumerate(PROMPT_OBJECTS):
    for seed in SEEDS:
        fname = f"prompt{i:02d}_seed{seed}"
        path = MIRROR_DIR / f"{fname}.png"
        if not path.exists():
            print(f"  Skipping (not found): {fname}")
            continue
        image = Image.open(path)
        obj_roi, ref_roi = get_object_and_reflection_rois(image, obj_name)
        all_rois[fname] = {
            "object": {"name": obj_roi.name, "token_indices": obj_roi.token_indices},
            "reflection": {"name": ref_roi.name, "token_indices": ref_roi.token_indices},
        }
        print(f"  {fname}: obj={obj_roi.size} tokens, ref={ref_roi.size} tokens")

with open(ROI_DIR / "clipseg_rois.json", "w") as f:
    json.dump(all_rois, f)
print(f"\nCLIPSeg ROIs saved to {ROI_DIR / 'clipseg_rois.json'}")

In [ ]:
# Preview generated images
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle("Sample Generated Images (seed=42)", fontsize=14)

for i in range(min(4, len(MIRROR_PROMPTS))):
    path = MIRROR_DIR / f"prompt{i:02d}_seed42.png"
    if path.exists():
        img = Image.open(path)
        axes[0, i].imshow(img)
        axes[0, i].set_title(f"Mirror {i}", fontsize=10)
    axes[0, i].axis("off")

for i in range(min(4, len(NON_MIRROR_PROMPTS))):
    path = NON_MIRROR_DIR / f"prompt{i:02d}_seed42.png"
    if path.exists():
        img = Image.open(path)
        axes[1, i].imshow(img)
        axes[1, i].set_title(f"Non-mirror {i}", fontsize=10)
    axes[1, i].axis("off")

plt.tight_layout()
plt.show()